# alpha.ipynb — rwrt Benchmark Suite

**Right Word at the Right Time** — adaptive Turkish vocabulary recommendation benchmark.

Evaluates how well each pipeline configuration recommends **semantically and morphologically
related** unknown words given a set of known words.

### Metrics

| Metric | What it measures |
|--------|------------------|
| **Stem-overlap ratio** | Fraction of top-K recommendations sharing a morphological root with ≥1 known word — higher means morphologically coherent suggestions |
| **Embedding coherence** | Mean cosine similarity between recommended words and the mean known-word embedding — higher means semantically closer to what the learner knows |

### Methods compared

| Method | Description |
|--------|-------------|
| `bi_only` | Bi-encoder + FAISS inner-product search |
| `bi_cross` | Bi-encoder retrieval + cross-encoder reranking |

Results are saved to `data/faiss_alpha/benchmark_results.json`.

---
**VRAM note:** designed for 8 GB GPUs. Models load one at a time; GPU memory is
cleared between experiments. Batch sizes are conservative.

In [12]:
import json
import math
import sys
import gc
import time
import itertools
import random
from pathlib import Path
from typing import Any
from collections import defaultdict

import faiss
import numpy as np
import torch

from rwrt.config import PipelineConfig
from rwrt.vocabulary import VocabularyStore
from rwrt.index import EmbeddingIndex
from rwrt.retrieve import BiEncoderRetriever
from rwrt.rerank import CrossEncoderReranker
from rwrt.types import Candidate
from rwrt.encoders import clear_encoder_cache

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

rng = random.Random(42)

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"FAISS: {faiss.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

Python: 3.12.7 (main, Mar 25 2026, 14:45:05) [GCC 15.2.1 20260123 (Red Hat 15.2.1-7)]
PyTorch: 2.11.0+cu130
FAISS: 1.13.2
CUDA available: True
Device: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.2 GB


In [13]:
# =====================================================================
# CONFIGURATION  —  edit this cell before running
# =====================================================================

# --- Paths ---
DB_PATH = Path("primary_graph.sqlite")
RESULTS_DIR = Path("data/faiss_alpha")
INDEX_BASE = Path("data/faiss")

# --- Vocabulary ---
MIN_WEIGHT = 10
MAX_VOCAB: int | None = 50_000       # None = use full vocabulary

# --- Learner simulation ---
# For each profile, we sample *profile_size* known words from across
# all frequency bands, then measure how semantically/morphologically
# related the recommendations are to those known words.
PROFILE_SIZES = [10, 25, 50, 100, 200, 500]
NUM_PROFILES_PER_SIZE = 5            # multiple random draws per size
RETURN_N = 10                         # evaluate at top 10

# --- Retrieval ---
RETRIEVE_K = 200                      # candidates from FAISS before reranking
ENCODE_BATCH_SIZE = 256
CROSS_BATCH_SIZE = 32

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FAISS_GPU = False                 # toggle FAISS GPU (not needed for 50k)

# --- Models ---
# Key: bi-encoder name, Value: whether to also test with cross-encoder
BI_ENCODERS: dict[str, bool] = {
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2": True,
    "sentence-transformers/distiluse-base-multilingual-cased-v2": True,
    "intfloat/multilingual-e5-small": False,
}
CROSS_ENCODER = "dbmdz/bert-base-turkish-cased"

# --- Query strategies to test ---
QUERY_STRATEGIES = [
    "mean_embedding",
    "weighted_mean",
    "inverse_weighted_mean",
]

# --- Topic exploration ---
TOPIC_KEYWORDS = ["yemek", "seyahat", "okul", "aile", "iş"]

# --- Misc ---
FORCE_REBUILD = False                 # rebuild all indices even if cached

# --- Derived ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INDEX_BASE.mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")
print(f"  Device:           {DEVICE}")
print(f"  Bi-encoders:      {list(BI_ENCODERS.keys())}")
print(f"  Query strategies: {QUERY_STRATEGIES}")
print(f"  Profile sizes:    {PROFILE_SIZES}")
print(f"  Draws per size:   {NUM_PROFILES_PER_SIZE}")

Configuration loaded.
  Device:           cuda
  Bi-encoders:      ['sentence-transformers/paraphrase-multilingual-mpnet-base-v2', 'sentence-transformers/distiluse-base-multilingual-cased-v2', 'intfloat/multilingual-e5-small']
  Query strategies: ['mean_embedding', 'weighted_mean', 'inverse_weighted_mean']
  Profile sizes:    [10, 25, 50, 100, 200, 500]
  Draws per size:   5


In [14]:
# =====================================================================
# Load vocabulary
# =====================================================================

cfg = PipelineConfig(
    db_path=DB_PATH,
    min_weight=MIN_WEIGHT,
    max_vocab_size=MAX_VOCAB,
    device=DEVICE,
    encode_batch_size=ENCODE_BATCH_SIZE,
    retrieve_k=RETRIEVE_K,
    return_n=RETURN_N,
    cross_batch_size=CROSS_BATCH_SIZE,
).resolve()

vocab = VocabularyStore(cfg)
vocab.load()
all_words = vocab.words()
freqs = [vocab.frequency(w) or 0 for w in all_words]

print(f"Vocabulary: {len(all_words):,} words")
print(f"  Min weight: {MIN_WEIGHT}")
print(f"  Max vocab:  {MAX_VOCAB or 'unlimited'}")
print(f"  Freq range: {min(freqs):,} – {max(freqs):,}")
print(f"  Top 10:     {all_words[:10]}")

Vocabulary: 50,000 words
  Min weight: 10
  Max vocab:  50000
  Freq range: 781 – 18,901,536
  Top 10:     ['bir', 'bu', 'ne', 've', 'için', 'mi', 'o', 'de', 'ben', 'çok']


In [15]:
# =====================================================================
# Metrics
# =====================================================================

def _common_prefix_ratio(w1: str, w2: str) -> float:
    """Fraction of the shorter word shared as a common prefix."""
    min_len = min(len(w1), len(w2))
    if min_len < 3:
        return 0.0
    common = 0
    for a, b in zip(w1.lower(), w2.lower()):
        if a == b:
            common += 1
        else:
            break
    return common / min_len


def stems_overlap(word: str, known_set: set[str]) -> bool:
    """Check if *word* shares a likely morphological root with any known word.

    Uses longest common prefix heuristic suited for agglutinative languages
    like Turkish (e.g. "gitmek" / "gidiyorum" share the root "git").
    """
    for known in known_set:
        ratio = _common_prefix_ratio(word, known)
        if ratio >= 0.4:
            return True
    return False


def stem_overlap_ratio(
    recommended: list[str],
    known_set: set[str],
    k: int = 10,
) -> float:
    """Fraction of top-K recommendations that share a root with known words."""
    if not recommended:
        return 0.0
    top = recommended[:k]
    if not top:
        return 0.0
    overlap = sum(1 for w in top if stems_overlap(w, known_set))
    return overlap / len(top)


def embedding_coherence(
    recommended: list[str],
    known_set: set[str],
    index: Any,
    k: int = 10,
) -> float:
    """Mean cosine similarity between top-K recommendations and the
    mean known-word embedding."""
    top = recommended[:k]
    if not top or not known_set:
        return 0.0

    known_words_in_index = [w for w in known_set if w in index._word_to_idx]
    if not known_words_in_index:
        return 0.0

    try:
        known_vecs = np.stack([index.embedding_for(w) for w in known_words_in_index])
        centroid = known_vecs.mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-10

        scores = []
        for w in top:
            if w in index._word_to_idx:
                vec = index.embedding_for(w)
                sim = float(np.dot(vec, centroid))
                scores.append(sim)
        return sum(scores) / len(scores) if scores else 0.0
    except (KeyError, ValueError):
        return 0.0

In [16]:
# =====================================================================
# Profile simulation
# =====================================================================

# Divide vocabulary into frequency quartiles for stratified sampling
n = len(all_words)
q1 = all_words[:n // 4]
q2 = all_words[n // 4: n // 2]
q3 = all_words[n // 2: 3 * n // 4]
q4 = all_words[3 * n // 4:]
quartiles = [q1, q2, q3, q4]
print(f"Frequency quartile sizes: {[len(q) for q in quartiles]}")


def sample_profile(profile_size: int, rng: random.Random) -> set[str]:
    """Sample *profile_size* known words stratified across all frequency bands."""
    per_quartile = max(1, profile_size // 4)
    known: set[str] = set()
    for q in quartiles:
        pool = [w for w in q if w not in known]
        if not pool:
            continue
        taken = rng.sample(pool, min(per_quartile, len(pool)))
        known.update(taken)
    # Fill remaining if short
    remaining = [w for w in all_words if w not in known]
    rng.shuffle(remaining)
    need = profile_size - len(known)
    if need > 0 and remaining:
        known.update(remaining[:need])
    return known


# Generate all profiles once
profiles: list[dict[str, Any]] = []
for size in PROFILE_SIZES:
    for _ in range(NUM_PROFILES_PER_SIZE):
        known = sample_profile(size, rng)
        profiles.append({
            "profile_size": size,
            "known": known,
        })

print(f"Generated {len(profiles)} profiles:")
by_size = defaultdict(int)
for p in profiles:
    by_size[p["profile_size"]] += 1
for size, count in sorted(by_size.items()):
    print(f"  {size:>4} known words  x  {count} draws")

Frequency quartile sizes: [12500, 12500, 12500, 12500]
Generated 30 profiles:
    10 known words  x  5 draws
    25 known words  x  5 draws
    50 known words  x  5 draws
   100 known words  x  5 draws
   200 known words  x  5 draws
   500 known words  x  5 draws


In [17]:
# =====================================================================
# Single-model index builder (build once, cache per model)
# =====================================================================

def _model_slug(model_name: str) -> str:
    return model_name.replace("/", "_").replace("-", "_")


def build_or_load_index(
    bi_model: str,
    vocab: VocabularyStore,
    base_cfg: PipelineConfig,
    force_rebuild: bool = False,
) -> tuple[EmbeddingIndex, PipelineConfig]:
    """Build (or load cached) FAISS index for *bi_model*."""
    index_dir = INDEX_BASE / _model_slug(bi_model)
    model_cfg = PipelineConfig(
        db_path=base_cfg.db_path,
        index_dir=index_dir,
        min_weight=base_cfg.min_weight,
        max_vocab_size=base_cfg.max_vocab_size,
        bi_model=bi_model,
        device=base_cfg.device,
        encode_batch_size=base_cfg.encode_batch_size,
    ).resolve()

    index = EmbeddingIndex(model_cfg, vocab)
    if not index.is_built or force_rebuild:
        print(f"  Building index: {bi_model}...")
        t0 = time.time()
        index.build(save_embeddings=True)
        elapsed = time.time() - t0
        print(f"  Done ({elapsed:.1f}s). Index: {index_dir / 'index.faiss'}")
    else:
        print(f"  Loading cached index: {bi_model}...")
        index.load()

    return index, model_cfg

In [18]:
# =====================================================================
# Experiment runner
# =====================================================================

def run_experiment(
    bi_model: str,
    query_strategy: str,
    use_cross: bool,
    profiles: list[dict[str, Any]],
    vocab: VocabularyStore,
    base_cfg: PipelineConfig,
    force_rebuild: bool = False,
) -> list[dict[str, Any]]:
    """Run one experiment configuration over all profiles.

    Measures stem-overlap ratio and embedding coherence.
    """
    # Free previous models
    clear_encoder_cache()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    index, model_cfg = build_or_load_index(
        bi_model, vocab, base_cfg, force_rebuild=force_rebuild
    )

    model_cfg.query_strategy = query_strategy  # type: ignore[assignment]
    model_cfg.retrieve_k = RETRIEVE_K
    model_cfg.return_n = RETURN_N
    model_cfg.cross_batch_size = CROSS_BATCH_SIZE

    retriever = BiEncoderRetriever(model_cfg, vocab, index)
    cross_encoder = CrossEncoderReranker(
        model_cfg, vocabulary=vocab, index=index
    ) if use_cross else None

    results = []
    for p in profiles:
        known = p["known"]

        t0 = time.time()

        candidates = retriever.retrieve(known, k=RETRIEVE_K)

        # Bi-encoder only ranking
        candidates.sort(key=lambda c: c.final_score, reverse=True)
        bi_top = candidates[:RETURN_N]
        bi_words = [c.word for c in bi_top]

        # Cross-encoder reranking (optional)
        if use_cross and cross_encoder:
            cross_top = cross_encoder.rerank(known, candidates, n=RETURN_N)
            cross_words = [c.word for c in cross_top]
        else:
            cross_words = []

        elapsed = time.time() - t0

        bi_stem = stem_overlap_ratio(bi_words, known, RETURN_N)
        bi_embed = embedding_coherence(bi_words, known, index, RETURN_N)

        result: dict[str, Any] = {
            "profile_size": p["profile_size"],
            "elapsed_s": round(elapsed, 3),
            "bi_only": {
                "words": bi_words[:5],
                "stem_overlap": bi_stem,
                "embedding_coherence": bi_embed,
            },
        }

        if cross_words:
            cross_stem = stem_overlap_ratio(cross_words, known, RETURN_N)
            cross_embed = embedding_coherence(cross_words, known, index, RETURN_N)
            result["bi_cross"] = {
                "words": cross_words[:5],
                "stem_overlap": cross_stem,
                "embedding_coherence": cross_embed,
            }

        results.append(result)

    # Clean up
    clear_encoder_cache()
    del index, retriever, cross_encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

In [19]:
# =====================================================================
# Run all experiments
# =====================================================================

experiments = []
for bi_model, use_cross in BI_ENCODERS.items():
    for qs in QUERY_STRATEGIES:
        experiments.append({
            "bi_model": bi_model,
            "query_strategy": qs,
            "use_cross_encoder": use_cross,
        })

total = len(experiments)
print(f"Total experiment configs: {total}")

all_results: list[dict[str, Any]] = []

for i, exp in enumerate(experiments, 1):
    label = (
        f"[{i}/{total}] "
        f"{exp['bi_model'].split('/')[-1]:45} "
        f"qs={exp['query_strategy']:25} "
        f"cross={'yes' if exp['use_cross_encoder'] else 'no '}"
    )
    print(f"\n{'=' * 100}")
    print(f"  {label}")
    print(f"{'=' * 100}")

    try:
        per_profile = run_experiment(
            bi_model=exp["bi_model"],
            query_strategy=exp["query_strategy"],
            use_cross=exp["use_cross_encoder"],
            profiles=profiles,
            vocab=vocab,
            base_cfg=cfg,
            force_rebuild=FORCE_REBUILD,
        )
    except Exception as exc:
        print(f"  FAILED: {exc}")
        import traceback
        traceback.print_exc()
        continue

    for r in per_profile:
        r["bi_model"] = exp["bi_model"]
        r["query_strategy"] = exp["query_strategy"]
        r["use_cross_encoder"] = exp["use_cross_encoder"]
        all_results.append(r)

    # Quick per-profile summary
    for r in per_profile:
        bi_stem = r["bi_only"]["stem_overlap"]
        bi_embed = r["bi_only"]["embedding_coherence"]
        cross_str = ""
        if "bi_cross" in r:
            cross_str = f"  cross_stem={r['bi_cross']['stem_overlap']:.3f}  cross_embed={r['bi_cross']['embedding_coherence']:.3f}"
        print(f"    profile={r['profile_size']:>3}:  stem={bi_stem:.3f}  embed={bi_embed:.3f}{cross_str}")

Total experiment configs: 9

  [1/9] paraphrase-multilingual-mpnet-base-v2         qs=mean_embedding            cross=yes
  Loading cached index: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    profile= 10:  stem=0.100  embed=0.938  cross_stem=0.100  cross_embed=0.919
    profile= 10:  stem=0.100  embed=0.920  cross_stem=0.100  cross_embed=0.905
    profile= 10:  stem=0.000  embed=0.942  cross_stem=0.000  cross_embed=0.922
    profile= 10:  stem=0.300  embed=0.935  cross_stem=0.000  cross_embed=0.919
    profile= 10:  stem=0.000  embed=0.942  cross_stem=0.000  cross_embed=0.927
    profile= 25:  stem=0.300  embed=0.963  cross_stem=0.000  cross_embed=0.947
    profile= 25:  stem=0.100  embed=0.952  cross_stem=0.100  cross_embed=0.940
    profile= 25:  stem=0.200  embed=0.971  cross_stem=0.200  cross_embed=0.959
    profile= 25:  stem=0.200  embed=0.966  cross_stem=0.000  cross_embed=0.948
    profile= 25:  stem=0.300  embed=0.965  cross_stem=0.100  cross_embed=0.951
    profile= 50:  stem=0.300  embed=0.968  cross_stem=0.100  cross_embed=0.956
    profile= 50:  stem=0.300  embed=0.972  cross_stem=0.100  cross_embed=0.954
    profile= 50:  stem=0.400  embed=0.972  cross_ste

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    profile= 10:  stem=0.900  embed=0.912  cross_stem=0.000  cross_embed=0.910
    profile= 10:  stem=0.700  embed=0.835  cross_stem=0.000  cross_embed=0.917
    profile= 10:  stem=1.000  embed=0.690  cross_stem=0.100  cross_embed=0.691
    profile= 10:  stem=0.000  embed=0.931  cross_stem=0.000  cross_embed=0.925
    profile= 10:  stem=0.200  embed=0.877  cross_stem=0.100  cross_embed=0.869
    profile= 25:  stem=0.200  embed=0.949  cross_stem=0.100  cross_embed=0.942
    profile= 25:  stem=0.100  embed=0.947  cross_stem=0.000  cross_embed=0.941
    profile= 25:  stem=0.100  embed=0.938  cross_stem=0.000  cross_embed=0.925
    profile= 25:  stem=0.000  embed=0.956  cross_stem=0.000  cross_embed=0.941
    profile= 25:  stem=0.100  embed=0.933  cross_stem=0.200  cross_embed=0.937
    profile= 50:  stem=0.200  embed=0.877  cross_stem=0.100  cross_embed=0.865
    profile= 50:  stem=0.300  embed=0.932  cross_stem=0.200  cross_embed=0.939
    profile= 50:  stem=0.200  embed=0.951  cross_ste

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    profile= 10:  stem=0.000  embed=0.926  cross_stem=0.000  cross_embed=0.907
    profile= 10:  stem=0.000  embed=0.940  cross_stem=0.000  cross_embed=0.923
    profile= 10:  stem=0.000  embed=0.904  cross_stem=0.000  cross_embed=0.883
    profile= 10:  stem=0.100  embed=0.938  cross_stem=0.000  cross_embed=0.928
    profile= 10:  stem=0.000  embed=0.931  cross_stem=0.000  cross_embed=0.922
    profile= 25:  stem=0.100  embed=0.960  cross_stem=0.200  cross_embed=0.947
    profile= 25:  stem=0.100  embed=0.955  cross_stem=0.000  cross_embed=0.942
    profile= 25:  stem=0.000  embed=0.961  cross_stem=0.100  cross_embed=0.945
    profile= 25:  stem=0.100  embed=0.960  cross_stem=0.100  cross_embed=0.946
    profile= 25:  stem=0.000  embed=0.949  cross_stem=0.000  cross_embed=0.939
    profile= 50:  stem=0.100  embed=0.960  cross_stem=0.200  cross_embed=0.947
    profile= 50:  stem=0.200  embed=0.964  cross_stem=0.500  cross_embed=0.949
    profile= 50:  stem=0.400  embed=0.958  cross_ste

In [20]:
# =====================================================================
# Results summary table
# =====================================================================

if not all_results:
    print("No results collected.")
else:
    agg: dict[tuple[str, str, bool], list[dict]] = defaultdict(list)
    for r in all_results:
        key = (r["bi_model"], r["query_strategy"], r["use_cross_encoder"])
        agg[key].append(r)

    header = f"{'Model':50} {'Strat':25} {'Cross':6} {'Method':10} {'Stem':8} {'Embed':8} {'Profiles':9}"
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for (model, qs, cross), profiles_list in sorted(agg.items()):
        for method in ["bi_only", "bi_cross"]:
            stem_vals = []
            embed_vals = []
            for p in profiles_list:
                if method not in p or p[method] is None:
                    continue
                stem_vals.append(p[method]["stem_overlap"])
                embed_vals.append(p[method]["embedding_coherence"])

            if not stem_vals:
                continue

            avg_stem = sum(stem_vals) / len(stem_vals)
            avg_embed = sum(embed_vals) / len(embed_vals)

            slug = model.split("/")[-1][:48]
            cross_str = "yes" if cross else "no "
            print(f"{slug:50} {qs:25} {cross_str:6} {method:10} {avg_stem:.4f}  {avg_embed:.4f}  {len(stem_vals)}")

    print("=" * len(header))

    # Markdown table
    print("\n--- Markdown table ---\n")
    print(f"| {'Model':50} | {'Strat':25} | {'Cross':6} | {'Method':10} | {'Stem':8} | {'Embed':8} |")
    print(f"|{'-'*52}|{'-'*27}|{'-'*8}|{'-'*12}|{'-'*10}|{'-'*10}|")
    for (model, qs, cross), profiles_list in sorted(agg.items()):
        for method in ["bi_only", "bi_cross"]:
            stem_vals = []
            embed_vals = []
            for p in profiles_list:
                if method not in p or p[method] is None:
                    continue
                stem_vals.append(p[method]["stem_overlap"])
                embed_vals.append(p[method]["embedding_coherence"])
            if not stem_vals:
                continue
            avg_stem = sum(stem_vals) / len(stem_vals)
            avg_embed = sum(embed_vals) / len(embed_vals)
            slug = model.split("/")[-1][:48]
            cross_str = "yes" if cross else "no "
            print(f"| {slug:50} | {qs:25} | {cross_str:6} | {method:10} | {avg_stem:.4f} | {avg_embed:.4f} |")

Model                                              Strat                     Cross  Method     Stem     Embed    Profiles 
--------------------------------------------------------------------------------------------------------------------------
multilingual-e5-small                              inverse_weighted_mean     no     bi_only    0.2900  0.9483  30
multilingual-e5-small                              mean_embedding            no     bi_only    0.3267  0.9493  30
multilingual-e5-small                              weighted_mean             no     bi_only    0.7067  0.9368  30
distiluse-base-multilingual-cased-v2               inverse_weighted_mean     yes    bi_only    0.2767  0.9584  30
distiluse-base-multilingual-cased-v2               inverse_weighted_mean     yes    bi_cross   0.2833  0.9461  30
distiluse-base-multilingual-cased-v2               mean_embedding            yes    bi_only    0.3067  0.9609  30
distiluse-base-multilingual-cased-v2               mean_embedding     

In [21]:
# =====================================================================
# Save results to JSON
# =====================================================================

output_path = RESULTS_DIR / "benchmark_results.json"

save_results = []
for r in all_results:
    sr = {
        "bi_model": r["bi_model"],
        "query_strategy": r["query_strategy"],
        "use_cross_encoder": r["use_cross_encoder"],
        "profile_size": r["profile_size"],
        "elapsed_s": r["elapsed_s"],
    }
    for method in ["bi_only", "bi_cross"]:
        if method in r and r[method] is not None:
            sr[method] = {
                "words": r[method].get("words", [])[:5],
                "stem_overlap": r[method]["stem_overlap"],
                "embedding_coherence": r[method]["embedding_coherence"],
            }
    save_results.append(sr)

output_path.write_text(
    json.dumps(save_results, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Saved {len(save_results)} result rows to {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved 270 result rows to data/faiss_alpha/benchmark_results.json
  File size: 150.3 KB


---
## Topic-based recommendation examples

The `topic` query strategy searches FAISS near a topic keyword directly,
skipping known-word aggregation. Below are examples using the default bi-encoder.

In [22]:
# =====================================================================
# Topic-based recommendation examples
# =====================================================================

default_model = list(BI_ENCODERS.keys())[0]
print(f"Using model: {default_model}\n")

idx, idx_cfg = build_or_load_index(default_model, vocab, cfg)

for topic in TOPIC_KEYWORDS:
    query_vec = idx.encode_text(topic)
    hits = idx.search(query_vec, k=10)
    print(f"\n  Topic: {topic}")
    print(f"  {'Word':20} {'Score':8} {'Freq':8}")
    print(f"  {'-'*20} {'-'*8} {'-'*8}")
    for word, score in hits:
        freq = vocab.frequency(word) or 0
        print(f"  {word:20} {score:.4f} {freq:>8,}")

clear_encoder_cache()
del idx
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Using model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2

  Loading cached index: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Topic: yemek
  Word                 Score    Freq    
  -------------------- -------- --------
  yemek                1.0000  260,677
  yiyecek              0.9919   57,534
  yemekleri            0.9868   10,071
  yiyeceği             0.9864    2,461
  mat                  0.9840    4,350
  yemeklerden          0.9808    1,315
  yemekler             0.9790   10,487
  yemeklere            0.9786    1,048
  yiyecekler           0.9767    6,171
  yiyecekleri          0.9758    3,869

  Topic: seyahat
  Word                 Score    Freq    
  -------------------- -------- --------
  seyahat              1.0000   28,275
  seyahate             0.9917    4,328
  reisi                0.9911    2,344
  seyahatte            0.9813      869
  seyahati             0.9777    2,248
  seyahatine           0.9676      872
  gezintiye            0.9543    4,563
  yolculuklar          0.9501    8,784
  gezisi               0.9389    5,511
  yolculuğa            0.9354    8,714

  Topic: okul
  Word 

---
## Notes

- The evaluation uses **intrinsic** metrics (stem-overlap ratio + embedding coherence)
  since no labeled ground-truth data exists for Turkish vocabulary recommendation.
- **Stem-overlap ratio** measures morphological coherence using a longest-common-prefix
  heuristic suited for Turkish's agglutinative structure.
- **Embedding coherence** measures how close recommendations are to the centroid of
  known words in the embedding space.
- The `frequency` baseline from earlier drafts was removed — it trivially predicts
  the most frequent unknown words, which doesn't test semantic relevance.
- Cross-encoder reranking (`bi_cross`) adds ~500 MB VRAM and latency but may improve
  precision for larger candidate pools.
- Smaller models (e5-small) are faster but may miss subtle semantic relationships
  in Turkish.
- Topic-based retrieval ignores known words entirely — it searches near a topic
  keyword. Useful for deliberate practice on a specific theme.